# SD-01 Word Documents: Loader x Splitter x Retriever Strategy Comparison

> Goal: Measure which combinations of loader, splitter, and retriever actually
> work for Word-document RAG -- and which fail, and why.

```
Loader      : WordLoader (naive vs structured)
Splitter    : DocumentProcessor (character-count) vs none (atomic units)
Embedding   : fastembed / BAAI/bge-base-en-v1.5 (local, 768-dim)
Vector DB   : Chroma
Retrievers  : Similarity / MMR / Hybrid (dense + BM25)
Metric      : recall@5 (substring match)
```

Learn:

* Why naive paragraph extraction drops tables and kills accuracy
* Why character-count splitting shreds markdown tables
* Which retriever wins for identifier-heavy documents
* How local embedding eliminates rate-limit problems entirely

**WHAT:** Run every combination of {naive-char, structured-unit, structured-char}
x {similarity, mmr, hybrid} on 4 real Word documents, measure recall@5 against
14 hand-checked QA pairs, and show which methods are good, which are bad, and
what we learned the hard way.

**WHY:** Choosing a RAG strategy by gut feel is how you end up with 0.33 recall
on a document full of tables. Measuring the full matrix lets you see exactly
where each method breaks and why.

**WHAT TO EXPECT:** A runnable demo on one document, then the full 4-document
results as static data (the full matrix takes ~10 minutes of embedding).
The "Bad methods" section below tells the story of three failures we hit
before arriving at the working pipeline.

In [1]:
import os
import sys
from pathlib import Path

# Resolve the repo root relative to this notebook's directory.
# When executed from NoteBooks/Special-Docs-01-Word-Documents/,
# the repo root is two levels up.
REPO_ROOT = Path.cwd().parent.parent
DOCX_DIR = REPO_ROOT / "Data" / "SD-01-word"

# Add the repo root to sys.path so we can import the project's modules.
sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DOCX_DIR  : {DOCX_DIR}")
print(f"Exists    : {DOCX_DIR.exists()}")

# List the .docx files we expect to find.
expected = [
    "fcc-nationwide-eas-test-2021.docx",
    "docx4j-getting-started.docx",
    "epa-combustion-turbine-tsd.docx",
    "undp-bda-evaluation-2025.docx",
]
for name in expected:
    path = DOCX_DIR / name
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}: {name}")

REPO_ROOT : /mnt/work/linux/Practice/RAG
DOCX_DIR  : /mnt/work/linux/Practice/RAG/Data/SD-01-word
Exists    : True
  OK: fcc-nationwide-eas-test-2021.docx
  OK: docx4j-getting-started.docx
  OK: epa-combustion-turbine-tsd.docx
  OK: undp-bda-evaluation-2025.docx


In [2]:
import hashlib
import json
import tempfile

# Repo modules -- these live under the repo root, added to sys.path above.
from loaders.word import WordLoader
from retrieval.hybrid import HybridRetriever
from retrieval.mmr import MMRRetriever
from retrieval.similarity import SimilarityRetriever
from splitters.recursive import DocumentProcessor
from vectordb.chroma import ChromaVectorStore

print("All imports OK")

All imports OK


**WHAT:** A dictionary of 14 hand-checked QA pairs across 4 documents. Each
pair is `(question, expected_answer_substring)`. The expected answer must
appear (case-insensitive) in at least one retrieved chunk for recall@1.

**WHY:** We chose exact-value answers like `"79.9%"`, `"13,320"`, `"slf4j"`,
`"EPA-HQ-OAR-2023-0072"` because:

- They are unambiguous (no "it depends" answers)
- They test whether specific data survived loading, splitting, and embedding
- A strategy that drops tables will miss table answers; a strategy that shreds
  tables will miss cell values

In [3]:
# 14 hand-checked QA pairs across 4 Word documents.
# Answers are exact substrings that must appear in retrieved chunk text.
QA = {
    "fcc-nationwide-eas-test-2021": [
        ("What was the filing rate for radio broadcasters?", "79.9%"),
        ("How many radio broadcasters participated in the nationwide test?", "13,320"),
        ("What percentage of radio broadcasters successfully retransmitted the alert?", "87.0%"),
        ("What was the most commonly reported complication?", "Audio Quality Issues"),
    ],
    "docx4j-getting-started": [
        ("Which Java class represents a .docx document in docx4j?", "WordprocessingMLPackage"),
        ("Which docx4j series is the last to run under Java 1.8?", "8.x"),
        ("What does docx4j use for logging?", "slf4j"),
        ("Can docx4j handle legacy binary .doc files?", "binary"),
    ],
    "epa-combustion-turbine-tsd": [
        ("What is the ISO base load of the LM6000 PC gas turbine?", "46.6"),
        ("What is the efficiency of the LM6000 PF gas turbine?", "41.4%"),
        ("What is the Docket ID for this technical support document?", "EPA-HQ-OAR-2023-0072"),
    ],
    "undp-bda-evaluation-2025": [
        ("When was the final evaluation report prepared?", "September 2025"),
        ("Who prepared the evaluation report?", "M&N Consultancy"),
        ("What is the name of the project being evaluated?", "Bakenyezi"),
    ],
}

total_q = sum(len(v) for v in QA.values())
print(f"{len(QA)} documents, {total_q} questions")

4 documents, 14 questions


**WHAT:** Three strategies for turning a .docx into chunks, defined by
loader mode x splitter choice:

| Strategy | Loader | Splitter | What happens |
|---|---|---|---|
| `naive-char` | WordLoader(mode="naive") | DocumentProcessor(1000, 200) | Flatten paragraphs (drops tables), then character-count split |
| `structured-unit` | WordLoader(mode="structured") | None | Walk body in order, keep each heading/paragraph/table as one atomic chunk |
| `structured-char` | WordLoader(mode="structured") | DocumentProcessor(1000, 200) | Walk body in order, then character-count split the structural units |

**WHY:** The naive loader is the trap everyone falls into -- it looks like it
works until you measure it against table-heavy documents. The structured loader
fixes the loading problem; the choice between "unit" and "char" determines
whether you then re-split those units or keep them atomic.

In [4]:
def build_chunks(docname, strategy):
    """Build chunks for one (document, strategy) pair."""
    path = str(DOCX_DIR / f"{docname}.docx")
    if strategy == "naive-char":
        # Naive mode: flatten paragraphs, drop tables entirely.
        docs = WordLoader(path, mode="naive").load()
        # Then split on character counts.
        chunks = DocumentProcessor(chunk_size=1000, chunk_overlap=200).split_docs(docs)
    elif strategy == "structured-unit":
        # Structured mode: walk body in order, keep each block atomic.
        # No splitter -- each heading, paragraph, or table is one chunk.
        chunks = WordLoader(path, mode="structured").load()
    elif strategy == "structured-char":
        # Structured mode: walk body in order, then split on char counts.
        docs = WordLoader(path, mode="structured").load()
        chunks = DocumentProcessor(chunk_size=1000, chunk_overlap=200).split_docs(docs)
    else:
        raise ValueError(f"unknown strategy: {strategy!r}")
    return chunks

# Quick sanity check: build one strategy and show chunk counts.
for strat in ["naive-char", "structured-unit", "structured-char"]:
    chunks = build_chunks("fcc-nationwide-eas-test-2021", strat)
    print(f"  {strat:20} -> {len(chunks)} chunks")

  naive-char           -> 146 chunks
  structured-unit      -> 147 chunks
  structured-char      -> 160 chunks


**WHAT:** `LocalEmbedder` wraps `fastembed` with the `BAAI/bge-base-en-v1.5`
model (768 dimensions, runs on CPU). Two operations:

- `embed_unique(texts)` -- deduplicates texts, embeds each unique text once,
  returns a `{sha1_hash: vector}` dict
- `embed_query(text)` -- embeds a single query string

**WHY:** This is the result of the first bad method we hit (see "Bad methods"
section below). The Gemini free tier limits you to ~100 embed requests/minute,
and every retry burns more quota, creating a retry storm that can never recover.
Local embedding has zero rate limits, no API key, and embeds 100 chunks in
~0.3s on CPU.

**WHAT TO EXPECT:** A `LocalEmbedder` instance, then a 768-dimensional vector
from a sample query.

In [5]:
class LocalEmbedder:
    """Local ONNX embedder (fastembed / BAAI/bge-base-en-v1.5, 768-dim).

    No API key, no rate limits: 100 chunks embed in ~0.3s on CPU.
    Uses the bge query prefix for queries and plain text for documents
    (fastembed handles this split internally via query_embed vs embed).
    """
    MODEL = "BAAI/bge-base-en-v1.5"

    def __init__(self):
        from fastembed import TextEmbedding
        self._emb = TextEmbedding(model_name=self.MODEL)

    def embed_unique(self, texts):
        """Embed all texts; returns {sha1: vector} for each unique text."""
        seen = set()
        # Preserve insertion order, skip duplicates.
        uniq = [t for t in texts if not (t in seen or seen.add(t))]
        # Batch-embed all unique texts at once.
        vectors = list(self._emb.embed(uniq, batch_size=64))
        # Key by SHA-1 hash so the same text always maps to the same vector.
        return {
            hashlib.sha1(t.encode("utf-8")).hexdigest(): v.tolist()
            for t, v in zip(uniq, vectors)
        }

    def embed_query(self, text):
        """Embed a single query string. Returns a 768-dim list."""
        return next(self._emb.query_embed(text)).tolist()

# Instantiate -- first call downloads the model (~130 MB), subsequent calls are fast.
embedder = LocalEmbedder()

# Quick test: embed a sample query.
sample_vec = embedder.embed_query("What was the filing rate?")
print(f"Embedding dimension: {len(sample_vec)}")
print(f"First 5 values: {sample_vec[:5]}")

/home/magus/.pyenv/versions/3.12.7/envs/magus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding dimension: 768
First 5 values: [-0.018479149788618088, -0.047340523451566696, 0.010677346959710121, -0.013462083414196968, 0.02065850980579853]


**WHAT:** Three retrieval strategies, all sharing one precomputed query vector:

| Retriever | How it works |
|---|---|
| similarity | Plain cosine distance between query vector and chunk vectors |
| mmr | Maximum Marginal Relevance -- trades pure relevance for diversity (lambda_mult=0.7) |
| hybrid | Reciprocal Rank Fusion of dense (vector) + sparse (BM25 keyword) rankings |

**WHY:** Similarity is the baseline. MMR prevents retrieving 5 nearly-identical
chunks. Hybrid adds BM25 keyword matching, which catches exact identifiers
(class names, library names, docket numbers) that dense embeddings miss.

**WHAT TO EXPECT:** Adapter classes that bridge the harness's precomputed vectors
into each retriever, then a function that runs all three on one question.

In [6]:
def _prevec_dense(store, query_vec):
    """Dense retriever adapter that reuses a precomputed query vector."""
    class _Adapter:
        def retrieve(self, question):
            return store.query(query_vec, top_k=5)
    return _Adapter()


def retrieve_variants(store, chunks, question, query_vec):
    """Run all 3 retrievers on one question, sharing one query vector.

    Similarity and MMR use the vector store directly.
    Hybrid fuses dense results with BM25 sparse results via RRF.
    """
    # Similarity: plain cosine top-5.
    sim = store.query(query_vec, top_k=5)

    # MMR: diverse top-5 with lambda_mult=0.7 (lean toward relevance).
    mmr = store.query_mmr(query_vec, top_k=5, lambda_mult=0.7)

    # Hybrid: dense + BM25 sparse, fused with Reciprocal Rank Fusion.
    try:
        from langchain_community.retrievers import BM25Retriever

        # Build a BM25 index over the chunks for this strategy.
        sparse = BM25Retriever.from_documents(chunks, k=5)

        # BM25Retriever is a langchain Runnable -- its .invoke() method
        # returns results. We wrap it in an adapter exposing .retrieve()
        # so HybridRetriever can use it uniformly.
        class _SparseAdapter:
            """BM25Retriever is a Runnable: expose .retrieve() via .invoke()."""
            def retrieve(self, q):
                return sparse.invoke(q)

        hyb = HybridRetriever(
            _prevec_dense(store, query_vec), _SparseAdapter(), top_k=5
        ).retrieve(question)
    except Exception as exc:
        print(f"    hybrid unavailable: {exc}")
        hyb = []

    return {"similarity": sim, "mmr": mmr, "hybrid": hyb}

**WHAT:** `recall@5` -- a binary metric per question. The expected answer
substring (lowercased) must appear in at least one of the top-5 retrieved
chunks' `page_content`.

**WHY:** We use substring matching rather than LLM-judged answer quality
because:

- The 14 answers are exact values: `"79.9%"`, `"13,320"`, `"slf4j"`,
  `"EPA-HQ-OAR-2023-0072"`
- A substring match proves the answer *exists* in the retrieved context --
  the LLM would find it
- It is deterministic, fast, and reproducible -- no LLM calls needed for
  the metric

**WHAT TO EXPECT:** A function that returns True/False, then a quick sanity check.

In [7]:
def recall_at_k(retrieved, expected):
    """Did the expected substring appear in any retrieved chunk?

    Args:
        retrieved: list of Document objects from the retriever.
        expected: the expected answer string (e.g. "79.9%").

    Returns:
        True if expected.lower() appears in any chunk's page_content.lower().
    """
    low = expected.lower()
    return any(low in d.page_content.lower() for d in retrieved)


# Sanity check: "79.9%" must be findable in a chunk that contains it.
class _FakeDoc:
    def __init__(self, text):
        self.page_content = text

assert recall_at_k(
    [_FakeDoc("filing rate was 79.9% of radio broadcasters")],
    "79.9%"
)
assert not recall_at_k(
    [_FakeDoc("the filing rate was high")],
    "79.9%"
)
print("recall_at_k: sanity checks passed")

recall_at_k: sanity checks passed


**WHAT:** Run the full strategy x retriever matrix on the FCC document only
(4 questions x 3 strategies x 3 retrievers = 36 retrievals). This takes
about 10-20 seconds and shows the measurement loop in action.

**WHY:** The FCC doc is the smallest and fastest to process. Running the demo
lets you see the harness work end-to-end before looking at the full
4-document results.

**WHAT TO EXPECT:** A grid of recall scores for each
(strategy, retriever, question) combination.

In [8]:
# --- DEMO: run the full matrix on the FCC document only ---
import time

DEMO_DOC = "fcc-nationwide-eas-test-2021"
strategies = ["naive-char", "structured-unit", "structured-char"]
retriever_names = ["similarity", "mmr", "hybrid"]

print(f"Demo: {DEMO_DOC}")
print(f"  strategies : {strategies}")
print(f"  retrievers : {retriever_names}")
print(f"  questions  : {len(QA[DEMO_DOC])}")
print()

# Phase 1: build chunks for all strategies, collect unique texts.
all_chunks = {}
for strategy in strategies:
    all_chunks[strategy] = build_chunks(DEMO_DOC, strategy)

# Collect all unique chunk texts across strategies for one embedding pass.
unique_texts = []
seen = set()
for strategy in strategies:
    for c in all_chunks[strategy]:
        if c.page_content not in seen:
            seen.add(c.page_content)
            unique_texts.append(c.page_content)

print(f"Unique chunks to embed: {len(unique_texts)}")
t0 = time.time()
vec_by_key = embedder.embed_unique(unique_texts)
print(f"Embedded in {time.time() - t0:.1f}s")

# Phase 2: for each strategy, build a Chroma store with precomputed
# embeddings, then run all retrievers on all questions.
tmp = tempfile.mkdtemp(prefix="sd01_chroma_")
demo_results = {}

for strategy in strategies:
    chunks = all_chunks[strategy]
    # Build store with embedding=None since we pass precomputed vectors.
    store = ChromaVectorStore(
        collection_name=f"demo__{strategy}",
        persist_dir=tmp,
        embedding=None,
    )
    # Look up the precomputed vector for each chunk by its SHA-1 hash.
    embs = [
        vec_by_key[hashlib.sha1(c.page_content.encode("utf-8")).hexdigest()]
        for c in chunks
    ]
    store.add(chunks, embeddings=embs)
    demo_results[strategy] = {"n_chunks": len(chunks)}

    print(f"\n[{strategy}] {len(chunks)} chunks")
    for q, expected in QA[DEMO_DOC]:
        # Embed the question once, reuse the vector across retrievers.
        query_vec = embedder.embed_query(q)
        variants = retrieve_variants(store, chunks, q, query_vec)

        row = {"expected": expected}
        for rname, retrieved in variants.items():
            row[rname] = recall_at_k(retrieved, expected)

        demo_results[strategy][q] = row
        flags = " ".join(f"{r}={'1' if row[r] else '0'}" for r in retriever_names)
        print(f"  Q: {q[:58]}")
        print(f"     exp={expected!r:18} {flags}")

# Print the recall grid.
print("\n--- Recall grid (1=hit, 0=miss) ---")
print(f"{'strategy':20} {'sim':>4} {'mmr':>4} {'hyb':>4}")
for strategy in strategies:
    nq = len(QA[DEMO_DOC])
    sim = sum(demo_results[strategy][q]["similarity"] for q, _ in QA[DEMO_DOC])
    mmr = sum(demo_results[strategy][q]["mmr"] for q, _ in QA[DEMO_DOC])
    hyb = sum(demo_results[strategy][q]["hybrid"] for q, _ in QA[DEMO_DOC])
    print(f"{strategy:20} {sim/nq:4.2f} {mmr/nq:4.2f} {hyb/nq:4.2f}")

Demo: fcc-nationwide-eas-test-2021
  strategies : ['naive-char', 'structured-unit', 'structured-char']
  retrievers : ['similarity', 'mmr', 'hybrid']
  questions  : 4

Unique chunks to embed: 203
Embedded in 141.7s

[naive-char] 146 chunks
  Q: What was the filing rate for radio broadcasters?
     exp='79.9%'            similarity=1 mmr=1 hybrid=1
  Q: How many radio broadcasters participated in the nationwide
     exp='13,320'           similarity=0 mmr=0 hybrid=0
  Q: What percentage of radio broadcasters successfully retrans
     exp='87.0%'            similarity=1 mmr=1 hybrid=1


/tmp/ipykernel_30324/1664161659.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


  Q: What was the most commonly reported complication?
     exp='Audio Quality Issues' similarity=0 mmr=0 hybrid=0

[structured-unit] 147 chunks
  Q: What was the filing rate for radio broadcasters?
     exp='79.9%'            similarity=1 mmr=1 hybrid=1
  Q: How many radio broadcasters participated in the nationwide
     exp='13,320'           similarity=0 mmr=0 hybrid=0
  Q: What percentage of radio broadcasters successfully retrans
     exp='87.0%'            similarity=1 mmr=1 hybrid=1
  Q: What was the most commonly reported complication?
     exp='Audio Quality Issues' similarity=1 mmr=1 hybrid=1

[structured-char] 160 chunks
  Q: What was the filing rate for radio broadcasters?
     exp='79.9%'            similarity=1 mmr=1 hybrid=1
  Q: How many radio broadcasters participated in the nationwide
     exp='13,320'           similarity=0 mmr=0 hybrid=0
  Q: What percentage of radio broadcasters successfully retrans
     exp='87.0%'            similarity=1 mmr=1 hybrid=1
  Q: What 

**WHAT:** The complete results across all 4 documents. This run took about
10 minutes of embedding time (100% local, no API calls, no rate limits).

**WHY:** The demo above shows the measurement loop on one document. These are
the real numbers from the full benchmark, which we present as static data
rather than re-running 10 minutes of embedding. The commented-out cell below
shows the full-run code if you want to reproduce it.

### Per-document recall@5 table

```
Document                         Strategy          Sim   MMR   Hyb   Chunks
fcc-nationwide-eas-test-2021     naive-char        0.50  0.50  0.50  146
fcc-nationwide-eas-test-2021     structured-unit   0.75  0.75  0.75  147
fcc-nationwide-eas-test-2021     structured-char   0.75  0.75  0.75  160
docx4j-getting-started           naive-char        0.50  0.50  0.75  935
docx4j-getting-started           structured-unit   0.50  0.50  0.75  944
docx4j-getting-started           structured-char   0.50  0.50  0.75  946
epa-combustion-turbine-tsd       naive-char        0.33  0.33  0.33  156
epa-combustion-turbine-tsd       structured-unit   1.00  1.00  1.00  142
epa-combustion-turbine-tsd       structured-char   1.00  1.00  1.00  176
undp-bda-evaluation-2025         naive-char        0.00  0.00  0.00  334
undp-bda-evaluation-2025         structured-unit   0.00  0.00  0.00  338
undp-bda-evaluation-2025         structured-char   0.00  0.00  0.00  383
```

### Grand totals (all 14 questions)

```
Strategy          Retriever      Recall
naive-char        similarity     5/14 = 0.36
naive-char        mmr            5/14 = 0.36
naive-char        hybrid         6/14 = 0.43
structured-char   similarity     8/14 = 0.57
structured-char   mmr            8/14 = 0.57
structured-char   hybrid         9/14 = 0.64
structured-unit   similarity     8/14 = 0.57
structured-unit   mmr            8/14 = 0.57
structured-unit   hybrid         9/14 = 0.64
```

The structured strategies (unit and char) are tied at 0.57-0.64. Hybrid
consistently beats similarity and MMR by about 7 percentage points.

In [9]:
# Full 4-doc benchmark results (pre-computed). Stored as a dict so you can
# slice and analyze without re-running the embed loop.
FULL_RESULTS = {
    "fcc-nationwide-eas-test-2021": {
        "naive-char":       {"sim": 0.50, "mmr": 0.50, "hyb": 0.50, "n_chunks": 146},
        "structured-unit":  {"sim": 0.75, "mmr": 0.75, "hyb": 0.75, "n_chunks": 147},
        "structured-char":  {"sim": 0.75, "mmr": 0.75, "hyb": 0.75, "n_chunks": 160},
    },
    "docx4j-getting-started": {
        "naive-char":       {"sim": 0.50, "mmr": 0.50, "hyb": 0.75, "n_chunks": 935},
        "structured-unit":  {"sim": 0.50, "mmr": 0.50, "hyb": 0.75, "n_chunks": 944},
        "structured-char":  {"sim": 0.50, "mmr": 0.50, "hyb": 0.75, "n_chunks": 946},
    },
    "epa-combustion-turbine-tsd": {
        "naive-char":       {"sim": 0.33, "mmr": 0.33, "hyb": 0.33, "n_chunks": 156},
        "structured-unit":  {"sim": 1.00, "mmr": 1.00, "hyb": 1.00, "n_chunks": 142},
        "structured-char":  {"sim": 1.00, "mmr": 1.00, "hyb": 1.00, "n_chunks": 176},
    },
    "undp-bda-evaluation-2025": {
        "naive-char":       {"sim": 0.00, "mmr": 0.00, "hyb": 0.00, "n_chunks": 334},
        "structured-unit":  {"sim": 0.00, "mmr": 0.00, "hyb": 0.00, "n_chunks": 338},
        "structured-char":  {"sim": 0.00, "mmr": 0.00, "hyb": 0.00, "n_chunks": 383},
    },
}

# Grand totals: {strategy x retriever: (hits, total)}
GRAND_TOTALS = {
    ("naive-char", "similarity"): (5, 14),
    ("naive-char", "mmr"):       (5, 14),
    ("naive-char", "hybrid"):    (6, 14),
    ("structured-char", "similarity"): (8, 14),
    ("structured-char", "mmr"):       (8, 14),
    ("structured-char", "hybrid"):    (9, 14),
    ("structured-unit", "similarity"): (8, 14),
    ("structured-unit", "mmr"):       (8, 14),
    ("structured-unit", "hybrid"):    (9, 14),
}

print("FULL_RESULTS loaded:", len(FULL_RESULTS), "docs")
for (strat, ret), (hits, total) in sorted(GRAND_TOTALS.items()):
    print(f"  {strat:16} x {ret:10} {hits}/{total} = {hits/total:.2f}")

FULL_RESULTS loaded: 4 docs
  naive-char       x hybrid     6/14 = 0.43
  naive-char       x mmr        5/14 = 0.36
  naive-char       x similarity 5/14 = 0.36
  structured-char  x hybrid     9/14 = 0.64
  structured-char  x mmr        8/14 = 0.57
  structured-char  x similarity 8/14 = 0.57
  structured-unit  x hybrid     9/14 = 0.64
  structured-unit  x mmr        8/14 = 0.57
  structured-unit  x similarity 8/14 = 0.57


In [ ]:
# OPTIONAL: re-run the full 4-document benchmark (takes ~10 minutes).
# Uncomment and execute this cell to reproduce the numbers above.
#
# import time
#
# full_strategies = ["naive-char", "structured-unit", "structured-char"]
# full_retrievers = ["similarity", "mmr", "hybrid"]
# tmp_full = tempfile.mkdtemp(prefix="sd01_chroma_full_")
# full_results = {}
#
# for docname in QA:
#     print(f"\n{'='*72}\n### {docname}\n{'='*72}")
#     full_results[docname] = {}
#
#     # One embedding pass per document, shared by all strategies.
#     all_chunks_full = {}
#     for strategy in full_strategies:
#         all_chunks_full[strategy] = build_chunks(docname, strategy)
#     unique_texts_full = []
#     seen_full = set()
#     for strategy in full_strategies:
#         for c in all_chunks_full[strategy]:
#             if c.page_content not in seen_full:
#                 seen_full.add(c.page_content)
#                 unique_texts_full.append(c.page_content)
#     vec_by_key_full = embedder.embed_unique(unique_texts_full)
#     print(f"  embedded {len(unique_texts_full)} unique chunks")
#
#     for strategy in full_strategies:
#         chunks = all_chunks_full[strategy]
#         store = ChromaVectorStore(
#             collection_name=f"{docname}__{strategy}",
#             persist_dir=tmp_full,
#             embedding=None,
#         )
#         embs = [
#             vec_by_key_full[hashlib.sha1(c.page_content.encode("utf-8")).hexdigest()]
#             for c in chunks
#         ]
#         store.add(chunks, embeddings=embs)
#         full_results[docname][strategy] = {"n_chunks": len(chunks)}
#
#         for q, expected in QA[docname]:
#             query_vec = embedder.embed_query(q)
#             variants = retrieve_variants(store, chunks, q, query_vec)
#             row = {"expected": expected}
#             for rname, retrieved in variants.items():
#                 row[rname] = recall_at_k(retrieved, expected)
#             full_results[docname][strategy][q] = row
#
# # Print summary.
# print("\nSUMMARY")
# print(f"{'doc':28} {'strategy':16} {'sim':>4} {'mmr':>4} {'hyb':>4} {'n':>3}")
# for docname, dres in full_results.items():
#     for strategy, sres in dres.items():
#         n = sres.get("n_chunks", 0)
#         nq = sum(1 for v in sres.values() if isinstance(v, dict) and "similarity" in v)
#         sim = sum(v["similarity"] for v in sres.values() if isinstance(v, dict) and "similarity" in v)
#         mmr = sum(v["mmr"] for v in sres.values() if isinstance(v, dict) and "mmr" in v)
#         hyb = sum(v["hybrid"] for v in sres.values() if isinstance(v, dict) and "hybrid" in v)
#         print(f"{docname[:28]:28} {strategy:16} {sim/nq:4.2f} {mmr/nq:4.2f} {hyb/nq:4.2f} {n:>3}")

## Bad methods we hit first, and how we overcame them

This is the heart of the notebook. Three real failures, three real fixes.

---

### BAD #1: Gemini embedding free tier

We started with `gemini-embedding-2` via
`langchain_google_genai.GoogleGenerativeAIEmbeddings`. The free tier allows
approximately 100 embed requests per minute, each holding up to 100 texts.

The problem: when you exceed the quota, the API returns HTTP 429
`RESOURCE_EXHAUSTED` with a message like "Please retry in 40.96s". Both
`langchain_google_genai` and the raw `google.genai` client retry 429s
internally with tenacity. But every retry burns more of the same quota that
is already exhausted -- so the backoff can never succeed. The retries pile
up, each one pushing you further over the limit, creating a retry storm.

We discovered this when embedding 4 documents x 3 strategies worth of chunks:
hundreds of unique texts, each batched in groups of 100, across multiple API
calls. The 429s started immediately and the retries made them permanent.

**OVERCOME:** Local ONNX embedder via `fastembed` with `BAAI/bge-base-en-v1.5`.
No API key, no rate limits, 768 dimensions, and 100 chunks embed in
approximately 0.3 seconds on CPU. We embed every unique chunk text once
(SHA-1 keyed dict), then pass precomputed vectors to Chroma. Zero network
calls during the entire benchmark.

---

### BAD #2: Naive loader drops tables

The `WordLoader(mode="naive")` calls `doc.paragraphs` from python-docx, which
only returns body-level paragraphs. Text inside table cells lives in a
different part of the XML tree, so flattening
`"\n".join(p.text for p in doc.paragraphs)` silently discards every table.

On the EPA combustion turbine document, this is catastrophic. The three
questions ask for values that only exist in tables: ISO base load "46.6",
efficiency "41.4%", and docket "EPA-HQ-OAR-2023-0072". With the naive
loader, none of these values exist in the corpus at all. Recall: 0.33 (only
1 of 3 questions answered, from a paragraph). With the structured loader,
all three are found. Recall: 1.00.

**OVERCOME:** `WordLoader(mode="structured")` walks
`doc.element.body.iterchildren()` in document order, which is the only
reliable way to see paragraphs and tables interleaved. Tables are
re-serialized as markdown (header row + `| --- |` + data rows), so their
values become searchable text that embedding and retrieval can match.

---

### BAD #3: Character-count splitting shreds tables

Even with the structured loader, if you then run
`RecursiveCharacterTextSplitter` (chunk_size=1000, chunk_overlap=200), the
splitter cuts on character counts. A markdown table is just text to it --
the splitter can slice a table mid-row, separate a header from its values,
or merge parts of two different tables into one chunk.

This hurts the EPA doc less (structured-char still scores 1.00 because the
tables are small enough to fit in one chunk), but it hurts the FCC doc: the
table containing "13,320" has 146 rows and spans well over 1000 characters.
Character splitting breaks it apart, and the specific cell with "13,320"
ends up in a chunk ranked 13th, outside the top-5.

**OVERCOME:** The `structured-unit` strategy keeps each structural block
(heading, paragraph, table) as one atomic chunk -- no splitting at all. The
`structured-char` strategy splits but only at character boundaries within
already-atomic units, which at least prevents merging content from different
structural blocks. For the best results on table-heavy documents, atomic
units win.

## Findings and limitations

Four case studies from the full benchmark, all backed by real numbers.

### 1. Hybrid beats similarity and MMR on identifier-heavy documents

On `docx4j-getting-started`, similarity and MMR score 0.50 while hybrid
scores 0.75. The two questions that only hybrid answers involve exact
technical identifiers: `WordprocessingMLPackage` (a Java class name) and
`slf4j` (a logging library). Dense embedding vectors represent meaning, not
exact strings -- "logging framework" does not embed close to "slf4j". But
BM25's lexical matching catches exact term overlaps, and RRF fusion promotes
those chunks into the top-5.

**Lesson:** dense + sparse hybrid retrieval is the robust default, especially
for technical documents with domain-specific identifiers.

### 2. The EPA win is all about tables

The EPA combustion turbine document has the starkest naive-vs-structured gap:
0.33 vs 1.00. All three questions ask for values that live exclusively in
tables. The naive loader drops every table, so those values simply do not
exist in the corpus. The structured loader re-serializes tables as markdown,
making "46.6", "41.4%", and "EPA-HQ-OAR-2023-0072" searchable. This is not
a subtle difference -- it is the difference between finding the answer and
not finding it at all.

### 3. undp = 0.00 everywhere -- title pollution, not a bug

All three UNDP questions score 0.00 across every strategy and retriever. The
answers ("September 2025", "M&N Consultancy", "Bakenyezi") do exist in the
corpus -- we verified they are in the chunks. But they rank 23rd to 297th.
The problem: the ALL-CAPS cover page chunk "FINAL EVALUATION REPORT" is the
highest-similarity hit for every question (sim scores 0.872, 0.747, 0.730)
because the questions themselves contain "evaluation report". The cover chunk
crowds out the actual answers.

This is an honest limitation. No strategy or retriever variant we tested
rescues these answers. Possible remedies: exclude the cover page via
metadata filtering, use a larger top_k, or rewrite queries to avoid the
polluting term.

### 4. FCC "13,320" -- answer present but too deep

The table chunk containing "13,320" exists in the corpus but ranks 13th
(similarity score 0.624). The top-5 is filled by descriptive prose: the
heading (0.683), three paragraphs (0.682, 0.670, 0.666). The exact table
value is semantically less similar to the question than the surrounding
prose.

**Lesson:** exact table values need either a larger top_k or table-aware
retrieval (e.g., query tables separately by metadata), not just better
embedding.

## What you should notice

* **The naive loader's recall drop on the EPA doc (0.33 vs 1.00) is the
  single most important number in this notebook** -- it shows what you lose
  when you ignore document structure.
* **Character-count splitting does not always hurt** (EPA scores 1.00 either
  way because the tables fit in one chunk), but it creates fragility on
  larger tables.
* **Hybrid retrieval's advantage shows up specifically on identifier-heavy
  documents** (docx4j), not universally.
* **The UNDP 0.00 scores are not a bug** -- they are a real limitation of
  embedding-based retrieval when the cover page dominates similarity.
* **Local embedding made this entire benchmark possible:** 10 minutes of
  embedding, zero API calls, zero rate limits, fully reproducible.
* **Precomputed embeddings are an efficiency pattern worth keeping:** embed
  every unique chunk once, pass vectors to Chroma, and never re-embed the
  same text for a different strategy.
* **The structured-unit and structured-char strategies tie on recall** (both
  0.57-0.64), but structured-unit produces fewer chunks (142-338 vs 160-383)
  because it does not split atomic units. Fewer chunks means faster indexing
  and less storage.

## Exercises

1. **Run the demo with your own .docx.** Point `DOCX_DIR` at a folder
   containing your own Word documents, add questions to the `QA` dict, and
   re-run the demo cell. Which strategy wins for your content?

2. **Try larger top_k.** Change `top_k=5` to `top_k=10` or `top_k=20` in
   `retrieve_variants`. Does the FCC "13,320" answer surface? Does the UNDP
   "Bakenyezi" answer surface?

3. **Exclude the cover page.** Filter out chunks with metadata
   `type="heading"` containing "FINAL EVALUATION REPORT" before building
   the store. Does the UNDP recall improve?

4. **Swap the embedding model.** Try `BAAI/bge-small-en-v1.5` (384-dim) or
   `sentence-transformers/all-MiniLM-L6-v2` (384-dim) by changing
   `LocalEmbedder.MODEL`. How do the scores change?

5. **Add more QA pairs.** Pick a document you care about, write 5 questions
   with exact answers, add them to `QA`, and re-run. Which strategy-retriever
   combination wins for your domain?

6. **Compare structured-unit vs structured-char chunk counts.** For each
   document, print the difference. When does character splitting actually
   help (more granular retrieval) vs hurt (fragmenting tables)?